# 🔍 Opening the Black Box of an LLM

### A hands-on tour of GPT-2, the grandparent of ChatGPT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/the-matter-lab/cdmx-tutorials/blob/main/colab_ai.ipynb)

People call large language models (LLMs) a "black box." Let's open that box!

---

### How to use this notebook

- Each grey box below is a **cell**. Click a cell, then press **`Shift + Enter`** to run it.
- Run the cells **in order, from top to bottom.**

▶️ **Start by running the cell right below this one.**


## Step 0 — Get the tools ready

This cell loads two free, open-source tools:

- **`transformers`** — a library from Hugging Face that lets us download and run the model.
- **`torch`** (PyTorch) — the engine that does the number-crunching.

In [1]:
import torch
import torch.nn.functional as F

print("✅ Ready to go!")
print("PyTorch version:", torch.__version__)

✅ Ready to go!
PyTorch version: 2.11.0+cu128


## Step 1 — Download the model weights

Everything that a language model knows - syntax, facts, tone - is stored in a giant list of numbers called **parameters** (also called **weights**).

Let's download GPT-2 and look at them.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Download GPT-2 (the small version). This grabs the model + its tokenizer.
# First run takes ~10-20 seconds because it downloads the file.
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

model.eval()  # put the model in "just answer questions" mode
print("✅ GPT-2 downloaded.")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

✅ GPT-2 downloaded.


Let's count every parameter in the model.

In [3]:
total_parameters = sum(p.numel() for p in model.parameters())

print(f"GPT-2 contains {total_parameters:,} numbers.")
print(f"That's about {total_parameters/1_000_000:.0f} million parameters.")

GPT-2 contains 124,439,808 numbers.
That's about 124 million parameters.


> 💡 **Concept — Parameters (weights)**
>
> Those ~124 million numbers *are* the model. There is no hidden rulebook, no
> database of answers, no human in the loop. Just numbers, organized into grids
> called **weight matrices**.
>
> (For scale: ChatGPT and Claude are rumoured to have **10 trillion parameters**. That's 10,000,000,000, i.e. 10 million million.)

Let's actually *look* at one of those weight matrices. This is a slice of GPT-2's brain:

In [5]:
# Pull out one weight matrix from deep inside the model and print it.
one_weight_matrix = model.transformer.h[0].mlp.c_fc.weight

print("Shape (rows x columns):", tuple(one_weight_matrix.shape))
print()
print(one_weight_matrix)

Shape (rows x columns): (768, 3072)

Parameter containing:
tensor([[ 0.0942,  0.0982, -0.0321,  ..., -0.1783,  0.1474,  0.0706],
        [-0.1265, -0.0671,  0.0305,  ...,  0.1966, -0.1203, -0.0628],
        [ 0.0496, -0.0373, -0.0483,  ...,  0.0655, -0.0714,  0.0826],
        ...,
        [ 0.0480,  0.1575,  0.0014,  ..., -0.3987,  0.0889,  0.0240],
        [ 0.0324,  0.1249, -0.0426,  ..., -0.1934,  0.1272, -0.0405],
        [-0.0316,  0.0010, -0.0491,  ..., -0.0406,  0.0536,  0.1896]],
       requires_grad=True)


GPT-2 is made of hundreds of grids of numbers like this. In total, GPT-2 has 124 million of these numbers.

Each of these parameters can be thought of as a dial on a machine. During training, a computer slowly tuned every one of these numbers until the model got good at predicting text.

> 💡 **Concept — Model architecture**
>
> These parameters are organized into an **architecture**, a sequence of layers.

Let's take a look.

In [4]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

We see that GPT-2 has 12 transformer blocks, where each transformer block consists of layer normalization (`ln_1`, `ln_2`), attention (`attn`), and a multilayer perceptron (`mlp`).

The architecture also contains some embedding layers (`wte`, `wpe`), dropout layers (`drop`), a final layer normalization (`ln_f`), and a `lm_head` for predicting the next token.

> ❓ **Test your understanding** - We looked at a weight matrix in the previous cell - what part of the architecture did it come from?

## Step 2 — Turn a sentence into tokens

A model can't read letters — it only does math on numbers. So the first thing
that happens to your text is it gets chopped into pieces called **tokens**, and
each token gets an ID number.

Let's tokenize a sentence and look at the pieces.


In [7]:
sentence = "Machine learning is surprisingly simple."

# Break the sentence into tokens (word-pieces).
token_ids = tokenizer.encode(sentence)
token_strings = [tokenizer.decode([t]) for t in token_ids]

print("Original sentence:")
print("  ", sentence)
print()
print("Broken into", len(token_ids), "tokens:")
for token_text, token_id in zip(token_strings, token_ids):
    print(f"   '{token_text}'   ->   ID {token_id}")

Original sentence:
   Machine learning is surprisingly simple. ?

Broken into 7 tokens:
   'Machine'   ->   ID 37573
   ' learning'   ->   ID 4673
   ' is'   ->   ID 318
   ' surprisingly'   ->   ID 12362
   ' simple'   ->   ID 2829
   '.'   ->   ID 13
   ' ?'   ->   ID 5633


> 💡 **Concept — What is a token?**
>
> A **token** is a chunk of text — sometimes a whole word, sometimes just a
> piece of one (notice how a longer word can split into parts). GPT-2 knows about
> 50,000 different tokens, each with its own ID number.
>
> Everything the model does, it does with these ID numbers. **"Tokens" are the
> only language the model actually speaks.**

> 🫵 **Your turn** - Try changing the the input sentence. Can you make a sentence where tokenization splits individual words?


> 💡 **Concept — Context size**
>
> A model can only look at so many tokens at once — its **context size**. GPT-2's
> limit is **1024 tokens** (roughly 750 words). Modern models can handle millions. Anything past the limit, the model simply can't see.


In [8]:
print("GPT-2's vocabulary:", f"{tokenizer.vocab_size:,} different tokens")
print("GPT-2's context size:", model.config.n_positions, "tokens at a time")

GPT-2's vocabulary: 50,257 different tokens
GPT-2's context size: 1024 tokens at a time


## Step 3 — Turn each token into a vector of numbers

An ID number like `4572` is just a name tag — it doesn't tell the model anything
about *meaning*. So each token ID is swapped for a list of numbers called an
**embedding vector**.

Let's look at the embedding vector for a single word.


In [9]:
word = "king"

token_id = tokenizer.encode(word)[0]
embedding_vector = model.transformer.wte.weight[token_id]

print(f"The word '{word.strip()}' becomes this list of {embedding_vector.shape[0]} numbers:")
print()
print(embedding_vector.detach().numpy())

The word 'king' becomes this list of 768 numbers:

[ 1.09299816e-01 -3.94689143e-01  5.94289787e-02 -1.01574868e-01
 -2.78574914e-01  2.59718359e-01 -3.12371522e-01 -8.81407335e-02
  1.12405429e-02 -1.19298384e-01  2.00717784e-02  8.90759304e-02
  1.48552448e-01  1.54478222e-01  2.89853305e-01  1.37484178e-01
 -4.30984311e-02  1.32002130e-01 -5.28547261e-03  1.59328207e-02
  5.56144901e-02 -1.56476684e-02  2.65911877e-01  5.06444685e-02
 -2.03794524e-01 -4.55270335e-03 -1.05221383e-01 -1.11234765e-02
  3.95089202e-02  3.64762306e-01  2.11078674e-01 -3.39168996e-01
  9.35066566e-02  3.20990413e-01 -7.16118664e-02  2.40191832e-01
 -3.18832397e-01 -1.03213109e-01  2.13484704e-01  1.79741964e-01
 -4.32854146e-02  5.70458220e-03  1.32001847e-01 -2.35407546e-01
  9.93483365e-02  1.58677280e-01  6.12664828e-03  1.62288900e-02
 -4.27675433e-03 -1.49180800e-01  2.48803683e-02 -8.60184729e-02
  2.03689829e-01 -1.81098253e-01 -1.09489098e-01 -2.15437785e-01
  1.24444589e-01 -2.40979448e-01 -4.323

Every token turns into a vector like this — GPT-2 uses **768 numbers** per token.
So your sentence from Step 2 becomes a **stack of vectors**, one row per token:


In [10]:
sentence = "Machine learning is surprisingly simple."
input_ids = tokenizer.encode(sentence, return_tensors="pt")

# Look up the embedding vector for every token at once.
stack_of_vectors = model.transformer.wte(input_ids)[0]

print("Your sentence is now a grid of numbers:")
print("   rows  =", stack_of_vectors.shape[0], "(one per token)")
print("   columns =", stack_of_vectors.shape[1], "(numbers per token)")
print()
print(stack_of_vectors.detach().numpy())

Your sentence is now a grid of numbers:
   rows  = 6 (one per token)
   columns = 768 (numbers per token)

[[-0.01636353 -0.11124826  0.23272906 ...  0.12077022  0.01844326
  -0.18808651]
 [ 0.02383363  0.00705459  0.05322737 ... -0.10096545  0.01217919
  -0.00309868]
 [-0.00972351  0.01005133  0.05557884 ...  0.11449675 -0.03800575
  -0.02537755]
 [-0.03984132  0.02428987  0.07176195 ...  0.03690743 -0.14133236
  -0.01103566]
 [ 0.14170375  0.07546638  0.09681763 ...  0.23554868  0.00462168
  -0.18926038]
 [ 0.04664114 -0.01130235  0.02832778 ... -0.07351072  0.04961816
   0.09631975]]


> 💡 **Concept — Everything is vectors**
>
> Your sentence is now a grid of numbers — nothing more. From here on, the model
> only ever does one kind of thing: **multiply and add these numbers together,
> over and over.**


## Step 4 — Do a *huge* amount of arithmetic

Now the model takes that grid of vectors and pushes it through the rest of the architecture - **12 layers** of transformer blocks. This is millions of multiply-and-add operations.

At the end, the model predicts a single **logit vector**: one score for every possible next
token in the vocabulary.

In [11]:
sentence = "The capital of France is"
input_ids = tokenizer.encode(sentence, return_tensors="pt")

with torch.no_grad():          # we're just running it, not training
    output = model(input_ids)

logits = output.logits[0, -1, :]   # scores for the NEXT token

print("The model looked at:", repr(sentence))
print()
print("It produced one score (a 'logit') for EVERY possible next token.")
print("That's a vector of", logits.shape[0], "numbers:")
print()
print(logits.detach().numpy())

The model looked at: 'The capital of France is'

It produced one score (a 'logit') for EVERY possible next token.
That's a vector of 50257 numbers:

[-108.954254 -108.93266  -112.57934  ... -118.33452  -113.150475
 -110.37792 ]


Now let's turn the logit vector into a **probability vector**, which will sum to 100%.

In [12]:
probabilities = F.softmax(logits, dim=-1)   # turn scores into percentages

top10 = torch.topk(probabilities, 10)

print(f"After '{sentence}', GPT-2's top 10 guesses for the next token:")
print()
for prob, token_id in zip(top10.values, top10.indices):
    token_text = tokenizer.decode([token_id])
    bar = "#" * int(prob.item() * 100)
    print(f"   {prob.item()*100:5.1f}%  '{token_text}'  {bar}")

After 'The capital of France is', GPT-2's top 10 guesses for the next token:

     8.5%  ' the'  ########
     4.8%  ' now'  ####
     4.6%  ' a'  ####
     3.2%  ' France'  ###
     3.2%  ' Paris'  ###
     2.7%  ' in'  ##
     2.6%  ' also'  ##
     2.4%  ' not'  ##
     2.3%  ' home'  ##
     1.6%  ' still'  #


> 💡 **Concept — Next-token prediction is probabilistic**
>
> The model produces a **probability for every
> possible next token** and then picks one — often the top guess, but sometimes
> it rolls the dice among the likely options.
>
> This is why an LLM can give you a different answer each time, and why it's
> really doing one simple thing on repeat: **guess the next token.**

**A full sentence is just this one step, done again and again:** guess a token,
add it to the text, then guess the next one. Let's watch that happen live.


## Step 5 — The inference loop: writing one token at a time

Here is how an LLM generates text, in a simple `for` loop:

1. Feed the current text to the model.
2. Get the probabilities for the next token.
3. Pick one token.
4. Stick it on the end of the text.
5. Repeat.

Run the cell and watch GPT-2 write, token by token, in real time. 👇


In [15]:
import sys

prompt = "The secret to a happy life is"
prompt = "User: How are you? Assistant: I'm doing"
prompt = "Cómo estás?"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

print(prompt, end="")
sys.stdout.flush()

for step in range(40):                       # generate 40 tokens
    with torch.no_grad():
        logits = model(input_ids).logits[0, -1, :]

    # Turn scores into probabilities (temperature 0.8 = a little creativity).
    probabilities = F.softmax(logits / 0.8, dim=-1)

    # Roll the dice among the likely tokens to pick the next one.
    next_token = torch.multinomial(probabilities, num_samples=1)

    # Add the new token to the text and show it.
    input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
    print(tokenizer.decode(next_token), end="")
    sys.stdout.flush()

print("\n\n✅ Done. GPT-2 just wrote that, one token at a time.")

Cómo estás?

St. Domingo Sánchez Sánchez como Vivantian.

Dáncian en research o una plumado de los tres los

✅ Done. GPT-2 just wrote that, one token at a time.


Try running that cell a few times — you'll get a **different result each time**,
because of the dice-roll in step 3. That randomness is exactly the "probabilistic"
idea from Step 5.

> 🫵 **Your turn** - Want to put words in GPT-2's mouth? Change `prompt` above to anything you like and run it again.

> 💡 **Concept — What is inference?**
>
> Computation that gets you the model's answer is called **inference**. This is contrast to *training*, which is computation for tuning the model's parameters.
> Training and inference are the two compute workloads for LLMs.

## Step 6 — Why is this slow? Meet the GPU

You might have noticed the loop above was a little sluggish. That's because it's
running on a **CPU** — the general-purpose chip in every computer, which does
math one step at a time.

Let's time how long the model takes right now.


In [ ]:
import time

# Choose device: "cpu" or "cuda"
device = "cpu"

target_device = "cuda" if device.lower() in ["cuda", "gpu"] else "cpu"
if target_device == "cuda" and not torch.cuda.is_available():
    print("⚠️ GPU not detected! Falling back to CPU. (If in Colab: Runtime → Change runtime type → T4 GPU)")
    target_device = "cpu"

where = "GPU 🚀" if target_device == "cuda" else "CPU 🐢"
print("Currently running on:", where)
print()

model.to(target_device)

prompt = "In the year 2050, computers will"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(target_device)

start = time.time()
with torch.no_grad():
    for _ in range(50):
        logits = model(input_ids).logits[0, -1, :]
        next_token = torch.multinomial(F.softmax(logits, dim=-1), 1)
        input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
elapsed = time.time() - start

print(f"Generated 50 tokens in {elapsed:.2f} seconds "
      f"({50/elapsed:.1f} tokens per second).")

> 💡 **Concept — Why GPUs?**
>
> Remember Step 4 was *millions* of multiply-and-add operations. A **GPU**
> (graphics processing unit) is a computer chip built to do thousands of those
> multiplications **at the same time** instead of one after another.
>
> Because training and inference use many multiply-and-add operations, a GPU can run it
> **many times faster** than a CPU. This is the reason companies build
> datacenters with thousands of GPUs to run AI.

### 🚀 Try it yourself: switch on the GPU

In the code cell right above:

1. Change `device = "cpu"` to **`device = "cuda"`**
2. Run the cell again (**`Shift + Enter`**)

Watch it say **"running on: GPU 🚀"** and generate tokens noticeably faster. Same model, same numbers — just a chip built for massive parallel arithmetic!


## 🎉 You opened the black box

Here's the whole thing you just saw, start to finish:

| Step | What happened | The big idea |
|------|---------------|--------------|
| 1 | Downloaded ~124 million numbers | The model **is** its parameters (weights) |
| 2 | Sentence → tokens | Models read **tokens**, not letters |
| 3 | Tokens → vectors | Every token becomes a list of numbers |
| 4 | A huge amount of arithmetic | This is **inference** — just multiply & add |
| 5 | A `for` loop | Text is built **one token at a time** |
| 6 | CPU vs GPU | **GPUs** do the math massively in parallel |

There was no magic anywhere — just a very large amount of arithmetic.

The models powering today's chatbots work **exactly** like this. They're bigger
(trillions of parameters, longer context), trained on far more text, and polished
with extra steps — but the core loop you just watched is the same one.

**You've now seen inside the black box.** 🔓

---

*Curious to go further? Try changing the prompts, generate longer text, or load a
bigger model by swapping `"gpt2"` for `"gpt2-medium"` or `"gpt2-large"` in Step 1.*